In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import font_manager
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

# 设置全局字体为微软雅黑
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
sns.set(font='Microsoft YaHei', style='whitegrid', palette='muted')

# 读取数据
df = pd.read_csv('/home/mw/input/data_total74037403/地区角度数据集/地区.csv')

# 设定特征和标签
y = df['评价指数']
X = df.drop(columns=['评价指数', '省份']) if '省份' in df.columns else df.drop(columns=['评价指数'])

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 随机森林回归，防止过拟合
rf = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train, y_train)

# 预测与评估
y_pred = rf.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# 特征重要性
importances = rf.feature_importances_
feat_imp = pd.DataFrame({'特征': X.columns, '重要性': importances}).sort_values('重要性', ascending=False)

# 完整消融实验：依次去掉每个特征，记录模型性能
ablation_results = []
for feature in feat_imp['特征']:
    X_train_ab = X_train.drop(columns=[feature])
    X_test_ab = X_test.drop(columns=[feature])
    rf_ab = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
    rf_ab.fit(X_train_ab, y_train)
    y_pred_ab = rf_ab.predict(X_test_ab)
    mse_ab = mean_squared_error(y_test, y_pred_ab)
    r2_ab = r2_score(y_test, y_pred_ab)
    ablation_results.append({'消融特征': feature, 'MSE': mse_ab, 'R2': r2_ab})

ablation_df = pd.DataFrame(ablation_results).sort_values('MSE')

# 结果可视化
fig = plt.figure(constrained_layout=True, figsize=(22, 12))
gs = fig.add_gridspec(2, 2)

# 特征重要性条形图
ax0 = fig.add_subplot(gs[0, 0])
sns.barplot(x='重要性', y='特征', data=feat_imp.head(15), palette='viridis', ax=ax0)
ax0.set_title('Top 15特征对评价指数的重要性', fontsize=16)
ax0.set_xlabel('重要性', fontsize=13)
ax0.set_ylabel('特征', fontsize=13)

# 预测效果对比
ax1 = fig.add_subplot(gs[0, 1])
sns.scatterplot(x=y_test, y=y_pred, s=120, color='b', label='原模型', ax=ax1)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
ax1.set_xlabel('真实值', fontsize=13)
ax1.set_ylabel('预测值', fontsize=13)
ax1.set_title('模型预测效果', fontsize=16)
ax1.legend()

# 消融实验结果热力图
ax2 = fig.add_subplot(gs[1, :])
ablation_pivot = ablation_df.set_index('消融特征')[['MSE', 'R2']]
sns.heatmap(ablation_pivot.T, annot=True, fmt=".2f", cmap='YlGnBu', cbar=True, ax=ax2)
ax2.set_title('消融实验：去除单一特征对模型性能的影响', fontsize=16)
ax2.set_xlabel('消融特征', fontsize=13)
ax2.set_ylabel('指标', fontsize=13)

plt.suptitle('各省份特征对评价指数影响分析与消融实验', fontsize=20, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# 结果表格
result_table = pd.DataFrame({
    '模型': ['原模型'],
    'MSE': [mse],
    'R2': [r2]
})
display(feat_imp)
display(result_table)
display(ablation_df)

/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:81: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:81: UserWarning: This figure was using constrained_layout==True, but that is incompatible with subplots_adjust and or tight_layout: setting constrained_layout==False. 


<Figure size 1584x864 with 4 Axes>

,特征,重要性
20,规模以上工业企业有效发明专利数（件）,0.177863
32,国外技术引进合同数,0.116683
31,孵化器内企业总数,0.098955
22,国内专利授权数,0.090753
14,按收发货人所在地分的进出口总额（亿万）,0.069098
21,规模以上工业企业新产品开发经费支出(万元),0.068681
17,规模以上工业企业R&D经费（万元）,0.053217
18,规模以上工业企业专利申请数（件）,0.043415
8,地区生产总值（亿元）,0.039244
16,规模以上工业企业R&D人员全时当量(人年),0.036680


,模型,MSE,R2
0,原模型,21.531945,0.950503


,消融特征,MSE,R2
2,孵化器内企业总数,19.352321,0.955514
31,中部地区,19.999863,0.954025
19,第三产业的单位占比,20.815428,0.952150
24,人口数,21.255953,0.951138
16,第三产业占比,21.741351,0.950022
27,2023高等学校有效发明专利,22.116813,0.949159
25,2023高等学校R&D课题经费（万元）,22.130113,0.949128
23,2023高等学校R&D成员,22.380577,0.948553
21,专科及以上学历占比,22.418419,0.948466
30,西部地区,22.503483,0.948270
